#  Gold Layer - Location Dimension

The Location Dimension enriches taxi trip data with geographic information.

In the source dataset, pickup and dropoff locations are represented only by numeric location IDs. While these IDs are suitable for storage, they are not meaningful for business users.

This dimension maps each Location ID to its corresponding Zone, Borough, and Service Zone using the official NYC Taxi Zone Lookup dataset.

### Source

- Silver Layer (`taxi.silver.yellow_taxi`)
- NYC Taxi Zone Lookup CSV

### Target

`taxi.gold.dim_location`

In [0]:
from pyspark.sql import functions as F

lookup_df = spark.read.option("header", True).csv(
    "/Volumes/taxi/default/raw_files/taxi_zone_lookup.csv"
)

In [0]:
display(lookup_df)

In [0]:
lookup_df.printSchema()

In [0]:
dim_location = (
    lookup_df
        .withColumnRenamed("LocationID","location_key")
        .withColumnRenamed("service_zone","service_zone")
)

In [0]:
print("Rows :", dim_location.count())

print(
    "Distinct Keys :",
    dim_location.select("location_key").distinct().count()
)

In [0]:
display(
    dim_location.orderBy("location_key")
)

In [0]:
(
    dim_location.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("taxi.gold.dim_location")
)

In [0]:
display(
    spark.table("taxi.gold.dim_location")
)

In [0]:
%sql
DESCRIBE DETAIL taxi.gold.dim_location;